# Predicting Stellar Class
## Score: .96634


In [34]:
OVERNIGHT_MODE = False

In [35]:
import numpy as np
import pandas as pd

DATA_DIR = "playground-series-s6e6"
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")

print(f"train: {train.shape}    test: {test.shape}")
print()
print("dtypes:")
print(train.dtypes)
print()

# class balance drives how aggressively we need to reweight for balanced accuracy
print("class distribution (train):")
print(train["class"].value_counts(normalize=True).round(4).to_string())
print()

nulls = train.isna().sum()
print("nulls per column (train):")
print(nulls[nulls > 0].to_string() if nulls.any() else "  none")
print()

print("categorical cardinalities:")
for c in ["spectral_type", "galaxy_population"]:
    vals = sorted(train[c].unique().tolist())
    print(f"  {c}: {len(vals)} unique  ->  {vals}")
print()

print("numeric summary:")
train.drop(columns=["id"]).describe().T

train: (577347, 12)    test: (247435, 11)

dtypes:
id                     int64
alpha                float64
delta                float64
u                    float64
g                    float64
r                    float64
i                    float64
z                    float64
redshift             float64
spectral_type         object
galaxy_population     object
class                 object
dtype: object

class distribution (train):
class
GALAXY    0.6538
QSO       0.2029
STAR      0.1433

nulls per column (train):
  none

categorical cardinalities:
  spectral_type: 4 unique  ->  ['A/F', 'G/K', 'M', 'O/B']
  galaxy_population: 2 unique  ->  ['Blue_Cloud', 'Red_Sequence']

numeric summary:


,count,mean,std,min,25%,50%,75%,max
alpha,577347.0,181.616673,96.242941,0.011684,132.161499,188.681465,231.829693,359.999810
delta,577347.0,21.834654,18.933570,-17.966988,2.474097,21.484412,36.988310,79.158322
u,577347.0,22.441926,2.018135,-0.139225,20.977090,22.570222,23.869103,28.253263
g,577347.0,21.007273,1.795426,13.535483,19.865005,21.467820,22.292715,27.620208
r,577347.0,19.962811,1.648964,12.579407,18.820671,20.431153,21.164096,25.254499
i,577347.0,19.378911,1.580059,11.962781,18.306820,19.631642,20.608191,27.910853
z,577347.0,19.041136,1.584365,11.682803,17.973192,19.188598,20.162111,26.826867
redshift,577347.0,0.723135,0.810070,-0.009970,0.181052,0.497525,0.881390,7.010780


In [36]:
BASE_FEATURES = ["alpha", "delta", "u", "g", "r", "i", "z", "redshift", "spectral_type", "galaxy_population"]
CAT_FEATURES = ["spectral_type", "galaxy_population"]
TARGET = "class"
LABELS = ["GALAXY", "QSO", "STAR"]
label_to_idx = {l: i for i, l in enumerate(LABELS)}
idx_to_label = {i: l for l, i in label_to_idx.items()}

MAGNITUDE_COLS = ["u", "g", "r", "i", "z"]
SENTINELS = {-9999.0, -999.0, -99.0, 99.0, 999.0, 9999.0}

def clean_sentinels(df):
    df = df.copy()
    cleaned = 0
    for c in MAGNITUDE_COLS + ["redshift"]:
        mask = df[c].isin(SENTINELS) | (df[c] > 50) | (df[c] < -50)
        if mask.any():
            cleaned += int(mask.sum())
            df.loc[mask, c] = np.nan
    return df, cleaned

def add_color_features(df, extended=False):
    df = df.copy()
    df["u_g"] = df["u"] - df["g"]
    df["g_r"] = df["g"] - df["r"]
    df["r_i"] = df["r"] - df["i"]
    df["i_z"] = df["i"] - df["z"]
    df["u_r"] = df["u"] - df["r"]
    df["g_i"] = df["g"] - df["i"]
    df["u_z"] = df["u"] - df["z"]
    if extended:
        df["u_i"] = df["u"] - df["i"]
        df["g_z"] = df["g"] - df["z"]
        df["r_z"] = df["r"] - df["z"]
        df["mag_mean"] = df[MAGNITUDE_COLS].mean(axis=1)
        df["mag_std"] = df[MAGNITUDE_COLS].std(axis=1)
        df["log1p_redshift"] = np.sign(df["redshift"]) * np.log1p(np.abs(df["redshift"]))
        df["u_g_sq"] = df["u_g"] ** 2
        df["g_r_sq"] = df["g_r"] ** 2
    return df

train_fe, train_cleaned = clean_sentinels(train)
test_fe, test_cleaned = clean_sentinels(test)
print(f"sentinel cleaning -> train: {train_cleaned} cells nan'd    test: {test_cleaned} cells nan'd")

train_fe = add_color_features(train_fe, extended=OVERNIGHT_MODE)
test_fe = add_color_features(test_fe, extended=OVERNIGHT_MODE)

COLOR_FEATURES = ["u_g", "g_r", "r_i", "i_z", "u_r", "g_i", "u_z"]
EXTENDED_FEATURES = ["u_i", "g_z", "r_z", "mag_mean", "mag_std", "log1p_redshift", "u_g_sq", "g_r_sq"]
ALL_FEATURES = BASE_FEATURES + COLOR_FEATURES + (EXTENDED_FEATURES if OVERNIGHT_MODE else [])

for c in CAT_FEATURES:
    train_fe[c] = train_fe[c].astype("category")
    test_fe[c] = test_fe[c].astype("category")

X = train_fe[ALL_FEATURES]
y = train_fe[TARGET].map(label_to_idx).astype(int)
X_test = test_fe[ALL_FEATURES]

print(f"X: {X.shape}    y: {y.shape}    X_test: {X_test.shape}")
print(f"features ({len(ALL_FEATURES)}): {ALL_FEATURES}")

sentinel cleaning -> train: 0 cells nan'd    test: 0 cells nan'd
X: (577347, 17)    y: (577347,)    X_test: (247435, 17)
features (17): ['alpha', 'delta', 'u', 'g', 'r', 'i', 'z', 'redshift', 'spectral_type', 'galaxy_population', 'u_g', 'g_r', 'r_i', 'i_z', 'u_r', 'g_i', 'u_z']


In [37]:
RUN_OPTUNA = True
OPTUNA_N_TRIALS = 30
OPTUNA_N_SPLITS = 3
OPTUNA_SEED = 42

best_params_lgb = None

if RUN_OPTUNA:
    import optuna
    from optuna.samplers import TPESampler
    from optuna.pruners import MedianPruner
    import lightgbm as lgb
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import balanced_accuracy_score
    from sklearn.utils.class_weight import compute_sample_weight

    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def objective(trial):
        params = {
            "objective": "multiclass",
            "num_class": 3,
            "metric": "multi_logloss",
            "verbose": -1,
            "seed": OPTUNA_SEED,
            "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.08, log=True),
            "num_leaves": trial.suggest_int("num_leaves", 31, 255),
            "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 50, 500, log=True),
            "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 1.0),
            "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 1.0),
            "bagging_freq": trial.suggest_int("bagging_freq", 1, 10),
            "lambda_l1": trial.suggest_float("lambda_l1", 1e-3, 10.0, log=True),
            "lambda_l2": trial.suggest_float("lambda_l2", 1e-3, 10.0, log=True),
            "min_gain_to_split": trial.suggest_float("min_gain_to_split", 0.0, 1.0),
        }

        skf = StratifiedKFold(n_splits=OPTUNA_N_SPLITS, shuffle=True, random_state=OPTUNA_SEED)
        fold_scores = []
        for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
            X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
            y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
            w_tr = compute_sample_weight("balanced", y_tr)

            dtrain = lgb.Dataset(X_tr, y_tr, weight=w_tr, categorical_feature=CAT_FEATURES)
            dvalid = lgb.Dataset(X_va, y_va, categorical_feature=CAT_FEATURES, reference=dtrain)

            model = lgb.train(
                params,
                dtrain,
                num_boost_round=3000,
                valid_sets=[dvalid],
                callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)],
            )

            va_pred = model.predict(X_va, num_iteration=model.best_iteration)
            fold_scores.append(balanced_accuracy_score(y_va, va_pred.argmax(axis=1)))

            trial.report(float(np.mean(fold_scores)), step=fold)
            if trial.should_prune():
                raise optuna.TrialPruned()

        return float(np.mean(fold_scores))

    def log_callback(study, trial):
        if trial.value is not None:
            print(f"trial {trial.number+1}/{OPTUNA_N_TRIALS}: balanced_acc = {trial.value:.5f}    best = {study.best_value:.5f}")
        else:
            best = study.best_value if len(study.trials) > 0 and any(t.value is not None for t in study.trials) else float("nan")
            print(f"trial {trial.number+1}/{OPTUNA_N_TRIALS}: PRUNED    best = {best:.5f}")

    sampler = TPESampler(seed=OPTUNA_SEED)
    pruner = MedianPruner(n_startup_trials=5, n_warmup_steps=1)
    study = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner)

    print(f"running Optuna: {OPTUNA_N_TRIALS} trials with {OPTUNA_N_SPLITS}-fold CV per trial")
    study.optimize(objective, n_trials=OPTUNA_N_TRIALS, callbacks=[log_callback])

    print(f"\nbest balanced_acc: {study.best_value:.5f}")
    print("best params:")
    for k, v in study.best_params.items():
        if isinstance(v, float):
            print(f"  {k}: {v:.6f}")
        else:
            print(f"  {k}: {v}")

    best_params_lgb = study.best_params

running Optuna: 30 trials with 3-fold CV per trial
trial 1/30: balanced_acc = 0.96386    best = 0.96386
trial 2/30: balanced_acc = 0.96327    best = 0.96386
trial 3/30: balanced_acc = 0.96321    best = 0.96386
trial 4/30: balanced_acc = 0.96458    best = 0.96458
trial 5/30: balanced_acc = 0.96307    best = 0.96458
trial 6/30: balanced_acc = 0.96449    best = 0.96458
trial 7/30: balanced_acc = 0.96417    best = 0.96458
trial 8/30: balanced_acc = 0.96496    best = 0.96496
trial 9/30: balanced_acc = 0.96468    best = 0.96496
trial 10/30: balanced_acc = 0.96465    best = 0.96496
trial 11/30: balanced_acc = 0.96475    best = 0.96496
trial 12/30: balanced_acc = 0.96467    best = 0.96496
trial 13/30: balanced_acc = 0.96401    best = 0.96496
trial 14/30: balanced_acc = 0.96492    best = 0.96496
trial 15/30: balanced_acc = 0.96367    best = 0.96496
trial 16/30: balanced_acc = 0.96493    best = 0.96496
trial 17/30: balanced_acc = 0.96483    best = 0.96496
trial 18/30: balanced_acc = 0.96472    b

In [38]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.utils.class_weight import compute_sample_weight

if OVERNIGHT_MODE:
    N_SPLITS = 15
    SEEDS = [42, 1337, 2024]
    LEARNING_RATE = 0.02
    NUM_BOOST_ROUND = 15000
    EARLY_STOPPING_ROUNDS = 200
else:
    N_SPLITS = 5
    SEEDS = [42]
    LEARNING_RATE = 0.05
    NUM_BOOST_ROUND = 5000
    EARLY_STOPPING_ROUNDS = 100

TOTAL_FITS = N_SPLITS * len(SEEDS)

default_params = {
    "objective": "multiclass",
    "num_class": 3,
    "metric": "multi_logloss",
    "learning_rate": LEARNING_RATE,
    "num_leaves": 63,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.9,
    "bagging_freq": 5,
    "min_data_in_leaf": 200,
    "verbose": -1,
}

if "best_params_lgb" in globals() and best_params_lgb is not None:
    base_params = {**default_params, **best_params_lgb}
    print(f"using Optuna-tuned params: {best_params_lgb}\n")
else:
    base_params = default_params

oof_lgb = np.zeros((len(X), 3))
test_pred_lgb = np.zeros((len(X_test), 3))
fi_lgb = np.zeros(len(ALL_FEATURES))
fit_idx = 0

for seed in SEEDS:
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    seed_oof = np.zeros((len(X), 3))

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), start=1):
        fit_idx += 1
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        w_tr = compute_sample_weight("balanced", y_tr)

        dtrain = lgb.Dataset(X_tr, y_tr, weight=w_tr, categorical_feature=CAT_FEATURES)
        dvalid = lgb.Dataset(X_va, y_va, categorical_feature=CAT_FEATURES, reference=dtrain)

        params = {**base_params, "seed": seed}
        model = lgb.train(
            params,
            dtrain,
            num_boost_round=NUM_BOOST_ROUND,
            valid_sets=[dvalid],
            callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False), lgb.log_evaluation(0)],
        )

        va_pred = model.predict(X_va, num_iteration=model.best_iteration)
        seed_oof[va_idx] = va_pred
        test_pred_lgb += model.predict(X_test, num_iteration=model.best_iteration)
        fi_lgb += model.feature_importance(importance_type="gain")

        fold_score = balanced_accuracy_score(y_va, va_pred.argmax(axis=1))
        print(f"fold {fit_idx}/{TOTAL_FITS}: balanced_acc = {fold_score:.5f}    best_iter = {model.best_iteration}")

    oof_lgb += seed_oof

oof_lgb /= len(SEEDS)
test_pred_lgb /= TOTAL_FITS
fi_lgb /= TOTAL_FITS

lgb_oof_score = balanced_accuracy_score(y, oof_lgb.argmax(axis=1))
print(f"\nLightGBM OOF balanced_acc: {lgb_oof_score:.5f}")

using Optuna-tuned params: {'learning_rate': 0.022615830881769158, 'num_leaves': 143, 'min_data_in_leaf': 126, 'feature_fraction': 0.6393810079651482, 'bagging_fraction': 0.9466338697710561, 'bagging_freq': 2, 'lambda_l1': 3.34056093058776, 'lambda_l2': 0.10288794335939705, 'min_gain_to_split': 0.2804839361658043}

fold 1/5: balanced_acc = 0.96604    best_iter = 1767
fold 2/5: balanced_acc = 0.96530    best_iter = 1465
fold 3/5: balanced_acc = 0.96538    best_iter = 1150
fold 4/5: balanced_acc = 0.96498    best_iter = 1417
fold 5/5: balanced_acc = 0.96526    best_iter = 1147

LightGBM OOF balanced_acc: 0.96539


In [39]:
oof_cb = None
test_pred_cb = None

if OVERNIGHT_MODE:
    try:
        from catboost import CatBoostClassifier
        _cb_available = True
    except ImportError:
        print("CatBoost not installed. Run: pip install catboost")
        print("Skipping CatBoost stage.")
        _cb_available = False

    if _cb_available:
        CB_N_SPLITS = 5
        CB_SEEDS = [42]
        CB_TOTAL_FITS = CB_N_SPLITS * len(CB_SEEDS)

        cat_idx = [ALL_FEATURES.index(c) for c in CAT_FEATURES]

        X_cb = X.copy()
        X_test_cb = X_test.copy()
        for c in CAT_FEATURES:
            X_cb[c] = X_cb[c].astype(str)
            X_test_cb[c] = X_test_cb[c].astype(str)

        oof_cb = np.zeros((len(X), 3))
        test_pred_cb = np.zeros((len(X_test), 3))
        fit_idx = 0

        for seed in CB_SEEDS:
            skf = StratifiedKFold(n_splits=CB_N_SPLITS, shuffle=True, random_state=seed)
            for fold, (tr_idx, va_idx) in enumerate(skf.split(X_cb, y), start=1):
                fit_idx += 1
                X_tr, X_va = X_cb.iloc[tr_idx], X_cb.iloc[va_idx]
                y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

                model = CatBoostClassifier(
                    iterations=15000,
                    learning_rate=0.03,
                    depth=7,
                    l2_leaf_reg=3.0,
                    loss_function="MultiClass",
                    eval_metric="MultiClass",
                    auto_class_weights="Balanced",
                    early_stopping_rounds=200,
                    thread_count=-1,
                    random_seed=seed,
                    verbose=0,
                )
                model.fit(X_tr, y_tr, eval_set=(X_va, y_va), cat_features=cat_idx)

                va_pred = model.predict_proba(X_va)
                oof_cb[va_idx] += va_pred / len(CB_SEEDS)
                test_pred_cb += model.predict_proba(X_test_cb) / CB_TOTAL_FITS

                fold_score = balanced_accuracy_score(y_va, va_pred.argmax(axis=1))
                print(f"cb fold {fit_idx}/{CB_TOTAL_FITS}: balanced_acc = {fold_score:.5f}    best_iter = {model.tree_count_}")

        cb_oof_score = balanced_accuracy_score(y, oof_cb.argmax(axis=1))
        print(f"\nCatBoost OOF balanced_acc: {cb_oof_score:.5f}")

In [40]:
from itertools import product

if oof_cb is not None:
    best_w, best_blend_score = 1.0, balanced_accuracy_score(y, oof_lgb.argmax(axis=1))
    for w in np.linspace(0, 1, 21):
        blend = w * oof_lgb + (1 - w) * oof_cb
        s = balanced_accuracy_score(y, blend.argmax(axis=1))
        if s > best_blend_score:
            best_blend_score, best_w = s, w
    oof_blend = best_w * oof_lgb + (1 - best_w) * oof_cb
    test_pred_blend = best_w * test_pred_lgb + (1 - best_w) * test_pred_cb
    print(f"best LightGBM weight: {best_w:.2f}    blended OOF balanced_acc: {best_blend_score:.5f}")
else:
    oof_blend = oof_lgb
    test_pred_blend = test_pred_lgb
    print(f"no CatBoost blend    OOF balanced_acc: {balanced_accuracy_score(y, oof_blend.argmax(axis=1)):.5f}")

base_scaled_score = balanced_accuracy_score(y, oof_blend.argmax(axis=1))
best_scalars = (1.0, 1.0, 1.0)
best_scaled_score = base_scaled_score

grid = np.linspace(0.5, 2.0, 21)
for s1, s2 in product(grid, grid):
    scalars = np.array([1.0, s1, s2])
    scaled = oof_blend * scalars
    s = balanced_accuracy_score(y, scaled.argmax(axis=1))
    if s > best_scaled_score:
        best_scaled_score, best_scalars = s, (1.0, s1, s2)

scalars = np.array(best_scalars)
oof_final = oof_blend * scalars
test_pred_final = test_pred_blend * scalars

print(f"best class scalars: GALAXY={best_scalars[0]:.3f}  QSO={best_scalars[1]:.3f}  STAR={best_scalars[2]:.3f}")
print(f"OOF balanced_acc after scaling: {best_scaled_score:.5f}    (delta: {best_scaled_score - base_scaled_score:+.5f})")

no CatBoost blend    OOF balanced_acc: 0.96539
best class scalars: GALAXY=1.000  QSO=1.250  STAR=1.475
OOF balanced_acc after scaling: 0.96582    (delta: +0.00043)


In [41]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred_final = oof_final.argmax(axis=1)

cm = confusion_matrix(y, y_pred_final)
cm_df = pd.DataFrame(cm, index=[f"true_{l}" for l in LABELS], columns=[f"pred_{l}" for l in LABELS])
print("confusion matrix (counts):")
print(cm_df.to_string())
print()

cm_norm = cm / cm.sum(axis=1, keepdims=True)
cm_norm_df = pd.DataFrame(cm_norm, index=[f"true_{l}" for l in LABELS], columns=[f"pred_{l}" for l in LABELS])
print("row-normalized (diagonal = per-class recall):")
print(cm_norm_df.round(5).to_string())
print()

per_class_recall = np.diag(cm_norm)
print("per-class recall (mean = balanced accuracy):")
for label, r in zip(LABELS, per_class_recall):
    print(f"  {label}: {r:.5f}")
print(f"  mean:        {per_class_recall.mean():.5f}")
print()

off_diag = []
for i in range(len(LABELS)):
    for j in range(len(LABELS)):
        if i != j:
            off_diag.append((LABELS[i], LABELS[j], int(cm[i, j]), cm_norm[i, j]))
off_diag.sort(key=lambda t: -t[2])
print("top off-diagonal errors (true -> pred):")
for true_l, pred_l, count, rate in off_diag:
    print(f"  {true_l:>6} -> {pred_l:<6}  {count:>8}    ({rate:.4%} of true {true_l})")
print()

print("classification report:")
print(classification_report(y, y_pred_final, target_names=LABELS, digits=5))

if "fi_lgb" in globals():
    fi_df = pd.DataFrame({"feature": ALL_FEATURES, "gain": fi_lgb}).sort_values("gain", ascending=False).reset_index(drop=True)
    total_gain = fi_df["gain"].sum()
    fi_df["pct_of_total"] = (fi_df["gain"] / total_gain * 100).round(3)
    fi_df["cumulative_pct"] = fi_df["pct_of_total"].cumsum().round(3)
    print("LightGBM feature importance (gain, averaged across all folds):")
    print(fi_df.to_string(index=False))

    weak = fi_df[fi_df["pct_of_total"] < 0.5]["feature"].tolist()
    if weak:
        print(f"\nfeatures contributing <0.5% of total gain (drop candidates): {weak}")
else:
    print("feature importance not available -- rerun the LightGBM CV cell to populate fi_lgb")

confusion matrix (counts):
             pred_GALAXY  pred_QSO  pred_STAR
true_GALAXY       359222      5628      12630
true_QSO            1677    114285       1181
true_STAR           2065       397      80262

row-normalized (diagonal = per-class recall):
             pred_GALAXY  pred_QSO  pred_STAR
true_GALAXY      0.95163   0.01491    0.03346
true_QSO         0.01432   0.97560    0.01008
true_STAR        0.02496   0.00480    0.97024

per-class recall (mean = balanced accuracy):
  GALAXY: 0.95163
  QSO: 0.97560
  STAR: 0.97024
  mean:        0.96582

top off-diagonal errors (true -> pred):
  GALAXY -> STAR       12630    (3.3459% of true GALAXY)
  GALAXY -> QSO         5628    (1.4909% of true GALAXY)
    STAR -> GALAXY      2065    (2.4963% of true STAR)
     QSO -> GALAXY      1677    (1.4316% of true QSO)
     QSO -> STAR        1181    (1.0082% of true QSO)
    STAR -> QSO          397    (0.4799% of true STAR)

classification report:
              precision    recall  f1-score

In [42]:
pred_labels = np.array([idx_to_label[i] for i in test_pred_final.argmax(axis=1)])
submission = pd.DataFrame({"id": test["id"], "class": pred_labels})
submission.to_csv("submission.csv", index=False)

print(f"submission shape: {submission.shape}")
print()
print("predicted class distribution:")
print(submission["class"].value_counts(normalize=True).round(4).to_string())
print()
submission.head()

submission shape: (247435, 2)

predicted class distribution:
class
GALAXY    0.6298
QSO       0.2081
STAR      0.1621



,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY
